# Semana 6 · Caso práctico Python: DATAPREP 360
## Inspección, limpieza y transformación básica de datos

**Objetivo técnico:** construir una versión limpia y trazable de un dataset sin inventar información ni borrar automáticamente observaciones que podrían ser válidas.

**Producto:** `dataprep360_limpio_v1.csv` + `bitacora_limpieza.csv`.

> Regla central: el dataset original se conserva intacto. Todo tratamiento se ejecuta sobre una copia y deja evidencia.


## Caso
Una organización multisede consolidó registros operativos de marzo a agosto de 2026 provenientes de tres fuentes: sistema, formulario e importación. El archivo ya fue recibido y documentado, pero todavía no está listo para análisis.

El archivo contiene distintos tipos de problemas deliberadamente incluidos para practicar un flujo transferible a proyectos de ventas, agricultura, seguridad ciudadana, educación, finanzas, logística y otros contextos.

**Campos principales:** identificador, fecha, sede, categoría, canal, cantidad, valor monetario, duración, estado, código de referencia, fuente y observación.


In [ ]:
# OPCIÓN A: Colab - subir el CSV manualmente
# Descomenta estas líneas si trabajarás en Google Colab.
#
# from google.colab import files
# uploaded = files.upload()
# ARCHIVO = next(iter(uploaded))

# OPCIÓN B: archivo en la misma carpeta del notebook
ARCHIVO = "dataprep360_sucio.csv"
print("Archivo:", ARCHIVO)


In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Leemos el archivo SIN modificarlo.
df_raw = pd.read_csv(ARCHIVO, encoding="utf-8-sig")

print("Dimensiones recibidas:", df_raw.shape)
display(df_raw.head(8))


## 1. Preservar el original
No trabajaremos directamente sobre `df_raw`. Creamos una copia de trabajo para que cualquier cambio sea reversible y auditable.


In [ ]:
df = df_raw.copy()

print("¿El original y la copia tienen las mismas dimensiones?", df_raw.shape == df.shape)


## 2. Inspección estructural antes de limpiar
Un perfilado inicial debe responder, como mínimo:

- ¿Cuántas filas y columnas existen?
- ¿Cómo se llaman los campos?
- ¿Qué tipos detecta Python?
- ¿Qué columnas parecen numéricas pero llegaron como texto?
- ¿Qué representaciones de valores faltantes existen?
- ¿Hay duplicados exactos?
- ¿Qué categorías muestran variantes?


In [ ]:
print("FILAS:", df.shape[0])
print("COLUMNAS:", df.shape[1])

print("\nENCABEZADOS:")
print(df.columns.tolist())

print("\nTIPOS DETECTADOS:")
print(df.dtypes)

print("\nDUPLICADOS EXACTOS (antes de cualquier cambio):")
print(df.duplicated().sum())


In [ ]:
# Muestra categorías tal como llegaron.
for col in ["SEDE ", "Categoría", "Estado", "Fuente"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(20))


## 3. Normalizar los nombres de columnas
La transformación de encabezados no cambia los datos: cambia la forma de referirnos a ellos. Usaremos nombres consistentes en minúsculas, sin tildes, espacios ni símbolos.


In [ ]:
def normalizar_nombre_columna(nombre):
    nombre = str(nombre).strip().lower()
    nombre = "".join(
        ch for ch in unicodedata.normalize("NFKD", nombre)
        if not unicodedata.combining(ch)
    )
    nombre = nombre.replace("(s/)", "soles")
    nombre = re.sub(r"[^a-z0-9]+", "_", nombre)
    return nombre.strip("_")

df.columns = [normalizar_nombre_columna(c) for c in df.columns]

print(df.columns.tolist())


## 4. Unificar marcadores de ausencia
`""`, `NA`, `N/A`, `-`, `sin dato` y `NULL` representan la misma idea: **el valor no está disponible**.

No deben confundirse con cero. Cero es un valor; ausencia es falta de información.


In [ ]:
MARCADORES_NULOS = {"", "na", "n/a", "-", "--", "sin dato", "null", "none", "nan"}

def normalizar_nulo(x):
    if pd.isna(x):
        return pd.NA
    texto = str(x).strip()
    return pd.NA if texto.lower() in MARCADORES_NULOS else texto

columnas_texto = [
    "registro_id", "fecha_evento", "sede", "categoria", "canal",
    "estado", "codigo_referencia", "fuente", "observacion"
]

for col in columnas_texto:
    df[col] = df[col].map(normalizar_nulo)

faltantes = df.isna().sum().sort_values(ascending=False)
display(faltantes.to_frame("faltantes"))


### Decisión técnica sobre faltantes
En este caso **no imputamos automáticamente**. Primero convertimos todas las formas de ausencia a `NA`.

La imputación solo debe realizarse cuando existe una regla defendible para la variable y para el problema del proyecto. Rellenar por conveniencia puede fabricar información.


## 5. Estandarizar categorías conocidas
Una misma sede o categoría no debe fragmentarse porque alguien escribió `CENTRO`, ` centro ` o `Centro`.

Solo normalizamos equivalencias conocidas. Los valores desconocidos no se inventan.


In [ ]:
def token(x):
    if pd.isna(x):
        return pd.NA
    texto = str(x).strip().lower()
    texto = "".join(
        ch for ch in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(ch)
    )
    return re.sub(r"\s+", " ", texto)

MAPAS = {
    "sede": {"centro": "Centro", "norte": "Norte", "sur": "Sur"},
    "categoria": {
        "atencion": "Atención", "venta": "Venta",
        "incidencia": "Incidencia", "monitoreo": "Monitoreo"
    },
    "estado": {"cerrado": "Cerrado", "pendiente": "Pendiente", "en proceso": "En proceso"},
    "canal": {"web": "Web", "presencial": "Presencial", "telefono": "Teléfono", "app": "App"},
    "fuente": {"sistema": "Sistema", "formulario": "Formulario", "importacion": "Importación"},
}

for col, mapa in MAPAS.items():
    clave = df[col].map(token)
    canonico = clave.map(mapa)
    df[col] = canonico.where(canonico.notna(), df[col].astype("string").str.strip())

for col in ["sede", "categoria", "estado", "fuente"]:
    print(f"\n{col.upper()}")
    print(df[col].value_counts(dropna=False))


## 6. Convertir fechas de forma controlada
El archivo contiene varios formatos de fecha. En vez de confiar en una interpretación ambigua, probamos formatos explícitos.

Los textos imposibles o fechas inválidas se convierten a `NaT` y quedan registrados como incidencia.


In [ ]:
def convertir_fecha_mixta(x):
    if pd.isna(x):
        return pd.NaT
    texto = str(x).strip()
    for formato in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%Y/%m/%d"):
        try:
            return pd.to_datetime(texto, format=formato)
        except (ValueError, TypeError):
            pass
    return pd.NaT

df["fecha_evento"] = df["fecha_evento"].map(convertir_fecha_mixta)

print(df["fecha_evento"].dtype)
print("Fechas no interpretables:", df["fecha_evento"].isna().sum())


## 7. Convertir números que llegaron como texto
Las columnas cuantitativas contienen ejemplos como `S/ 120.50`, `15,8`, `24 und` o `36 min`.

Primero retiramos símbolos/unidades conocidos y luego usamos conversión numérica con `errors="coerce"` para no detener el proceso ante un valor problemático.


In [ ]:
def convertir_numero(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)

    texto = str(x).strip().lower()

    if texto in MARCADORES_NULOS:
        return np.nan

    texto = texto.replace(",", ".")
    texto = re.sub(r"(s/|pen|und|min)", "", texto)
    texto = re.sub(r"[^0-9.\-]+", "", texto)

    return pd.to_numeric(texto, errors="coerce")

for col in ["cantidad", "valor_soles", "duracion_min"]:
    df[col] = df[col].map(convertir_numero)

print(df[["cantidad", "valor_soles", "duracion_min"]].dtypes)
display(df[["cantidad", "valor_soles", "duracion_min"]].head(10))


## 8. Duplicados: exactos ≠ lógicos
- **Duplicado exacto:** la fila completa se repite.
- **Duplicado lógico:** dos filas parecen representar el mismo evento según una clave de negocio, aunque algún campo sea distinto.

Los exactos pueden retirarse cuando se confirma que son una repetición de carga. Los lógicos se **marcan** para revisión; no se borran por automatismo.


In [ ]:
duplicados_exactos = df.duplicated(keep="first")

print("Duplicados exactos:", duplicados_exactos.sum())

# Conservamos la primera aparición.
df = df.loc[~duplicados_exactos].copy()

clave_negocio = ["fecha_evento", "sede", "categoria", "codigo_referencia"]

df["flag_duplicado_logico"] = df.duplicated(
    subset=clave_negocio,
    keep=False
)

print("Filas involucradas en posibles duplicados lógicos:",
      df["flag_duplicado_logico"].sum())

display(
    df.loc[df["flag_duplicado_logico"],
           ["registro_id"] + clave_negocio + ["estado", "observacion"]]
    .sort_values(clave_negocio)
    .head(20)
)


## 9. Valores imposibles: marcar antes de corregir
En este caso conocemos reglas operativas mínimas:

- `cantidad` debe estar entre 0 y 500;
- `valor_soles` no puede ser negativo;
- `duracion_min` debe estar entre 0 y 1440;
- la fecha no puede ser futura respecto a la fecha de referencia de la práctica.

Cuando no podemos recuperar el valor verdadero desde la fuente, **no lo sustituimos por un número inventado**. Creamos indicadores (`flags`) para mantener la trazabilidad.


In [ ]:
HOY_REFERENCIA = pd.Timestamp("2026-09-15")

df["flag_fecha_invalida"] = (
    df["fecha_evento"].isna()
    | (df["fecha_evento"] > HOY_REFERENCIA)
)

df["flag_cantidad_imposible"] = (
    df["cantidad"].notna()
    & ((df["cantidad"] < 0) | (df["cantidad"] > 500))
)

df["flag_valor_imposible"] = (
    df["valor_soles"].notna()
    & (df["valor_soles"] < 0)
)

df["flag_duracion_imposible"] = (
    df["duracion_min"].notna()
    & ((df["duracion_min"] < 0) | (df["duracion_min"] > 1440))
)

columnas_flags = [
    "flag_fecha_invalida",
    "flag_cantidad_imposible",
    "flag_valor_imposible",
    "flag_duracion_imposible",
]

display(df[columnas_flags].sum().to_frame("registros_marcados"))


### Tratamiento de los valores demostrablemente inválidos
Una vez creada la bandera, el valor imposible se convierte a `NA` en la **copia de trabajo**. No lo reemplazamos por un valor inventado.

La evidencia no se pierde:
- el archivo original permanece intacto en `df_raw`;
- la bandera indica qué filas fueron afectadas;
- la bitácora documenta la regla aplicada.


In [ ]:
# Pasamos a NA únicamente los valores que violan reglas explícitas del caso.
df.loc[df["flag_fecha_invalida"], "fecha_evento"] = pd.NaT
df.loc[df["flag_cantidad_imposible"], "cantidad"] = np.nan
df.loc[df["flag_valor_imposible"], "valor_soles"] = np.nan
df.loc[df["flag_duracion_imposible"], "duracion_min"] = np.nan

print("Valores inválidos retirados de las columnas analíticas, con flags conservados.")


## 10. Posibles atípicos: detectar no significa eliminar
Aplicamos el criterio IQR como **bandera de revisión**, no como sentencia de borrado.

\[
IQR = Q_3 - Q_1
\]

Límites de exploración:

\[
L_i = Q_1 - 1.5(IQR), \qquad L_s = Q_3 + 1.5(IQR)
\]

Un dato fuera de esos límites puede ser error, pero también puede ser un evento real importante.


In [ ]:
def limites_iqr(serie):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    return q1, q3, q1 - 1.5 * iqr, q3 + 1.5 * iqr

validos_valor = df["valor_soles"].notna() & ~df["flag_valor_imposible"]
q1v, q3v, liv, lsv = limites_iqr(df.loc[validos_valor, "valor_soles"])

df["flag_valor_atipico"] = (
    validos_valor
    & ((df["valor_soles"] < liv) | (df["valor_soles"] > lsv))
)

validos_duracion = df["duracion_min"].notna() & ~df["flag_duracion_imposible"]
q1d, q3d, lid, lsd = limites_iqr(df.loc[validos_duracion, "duracion_min"])

df["flag_duracion_atipica"] = (
    validos_duracion
    & ((df["duracion_min"] < lid) | (df["duracion_min"] > lsd))
)

print(f"VALOR: Q1={q1v:.2f}, Q3={q3v:.2f}, límite inferior={liv:.2f}, límite superior={lsv:.2f}")
print(f"DURACIÓN: Q1={q1d:.2f}, Q3={q3d:.2f}, límite inferior={lid:.2f}, límite superior={lsd:.2f}")

print("\nPosibles atípicos en valor:", df["flag_valor_atipico"].sum())
print("Posibles atípicos en duración:", df["flag_duracion_atipica"].sum())


In [ ]:
# Revisamos algunos posibles atípicos, sin borrarlos.
display(
    df.loc[
        df["flag_valor_atipico"] | df["flag_duracion_atipica"],
        ["registro_id", "fecha_evento", "sede", "categoria",
         "valor_soles", "duracion_min",
         "flag_valor_atipico", "flag_duracion_atipica"]
    ].head(20)
)


## 11. Bitácora de limpieza
Una transformación que no queda documentada es difícil de auditar, repetir o defender.

La bitácora registra **qué problema se encontró, qué regla se aplicó, cuántos registros fueron afectados y por qué se tomó esa decisión**.


In [ ]:
bitacora = pd.DataFrame([
    {
        "etapa": "Estructura",
        "variable": "Encabezados",
        "problema": "Nombres con espacios, tildes y símbolos",
        "accion": "Normalizar a snake_case",
        "registros_afectados": len(df.columns),
        "justificacion": "Facilita un procesamiento reproducible y evita referencias ambiguas."
    },
    {
        "etapa": "Nulos",
        "variable": "Varias columnas",
        "problema": "Marcadores heterogéneos de ausencia",
        "accion": "Convertir a NA",
        "registros_afectados": int(df.isna().sum().sum()),
        "justificacion": "Unifica la ausencia sin inventar información."
    },
    {
        "etapa": "Duplicados",
        "variable": "Todas las columnas",
        "problema": "Filas completamente repetidas",
        "accion": "Eliminar duplicados exactos, conservando primera aparición",
        "registros_afectados": int(duplicados_exactos.sum()),
        "justificacion": "La repetición exacta no agrega información si proviene de una carga duplicada."
    },
    {
        "etapa": "Duplicados",
        "variable": "Clave de negocio",
        "problema": "Posibles eventos repetidos",
        "accion": "Marcar para revisión",
        "registros_afectados": int(df["flag_duplicado_logico"].sum()),
        "justificacion": "Un duplicado lógico requiere criterio del dominio; no se elimina automáticamente."
    },
    {
        "etapa": "Reglas básicas",
        "variable": "cantidad / valor_soles / duracion_min / fecha_evento",
        "problema": "Valores imposibles o no interpretables",
        "accion": "Marcar con flags y convertir a NA en la copia de trabajo",
        "registros_afectados": int(df[
            ["flag_fecha_invalida", "flag_cantidad_imposible",
             "flag_valor_imposible", "flag_duracion_imposible"]
        ].any(axis=1).sum()),
        "justificacion": "Sin una fuente confiable no se debe inventar el valor correcto; NA evita contaminar cálculos posteriores."
    },
    {
        "etapa": "Atípicos",
        "variable": "valor_soles / duracion_min",
        "problema": "Observaciones alejadas del patrón",
        "accion": "Marcar con IQR; conservar",
        "registros_afectados": int(
            df["flag_valor_atipico"].sum()
            + df["flag_duracion_atipica"].sum()
        ),
        "justificacion": "Atípico no equivale a error; puede contener información relevante."
    },
])

display(bitacora)


## 12. Resumen antes de exportar
Todavía no estamos haciendo análisis descriptivo del problema. Solo comprobamos que el tratamiento ejecutado produjo una **versión técnica de trabajo** y que las incidencias pendientes permanecen visibles.


In [ ]:
resumen = pd.DataFrame({
    "indicador": [
        "Filas recibidas",
        "Filas después de eliminar duplicados exactos",
        "Duplicados exactos eliminados",
        "Filas en duplicados lógicos marcadas",
        "Fechas inválidas/futuras marcadas",
        "Cantidades imposibles marcadas",
        "Valores negativos marcados",
        "Duraciones imposibles marcadas",
        "Posibles atípicos en valor",
        "Posibles atípicos en duración",
    ],
    "resultado": [
        len(df_raw),
        len(df),
        int(duplicados_exactos.sum()),
        int(df["flag_duplicado_logico"].sum()),
        int(df["flag_fecha_invalida"].sum()),
        int(df["flag_cantidad_imposible"].sum()),
        int(df["flag_valor_imposible"].sum()),
        int(df["flag_duracion_imposible"].sum()),
        int(df["flag_valor_atipico"].sum()),
        int(df["flag_duracion_atipica"].sum()),
    ]
})

display(resumen)


## 13. Exportar el producto de la Semana 6
Se generan dos archivos:

1. **Dataset limpio v1:** contiene tipos y formatos estandarizados, duplicados exactos retirados y banderas para incidencias que todavía necesitan revisión.
2. **Bitácora de limpieza:** permite demostrar qué se hizo y por qué.

> La validación formal del conjunto de datos corresponde a la siguiente etapa del proyecto.


In [ ]:
df.to_csv("dataprep360_limpio_v1.csv", index=False, encoding="utf-8-sig")
bitacora.to_csv("bitacora_limpieza.csv", index=False, encoding="utf-8-sig")

print("Archivos generados:")
print(" - dataprep360_limpio_v1.csv")
print(" - bitacora_limpieza.csv")


## 14. Cómo adaptar el pipeline a cualquier proyecto
No copies reglas de negocio que no correspondan. Conserva el **patrón de trabajo** y cambia las variables/reglas:

- **Agricultura:** fecha, lote, temperatura, humedad, conteo de plaga.
- **Seguridad:** fecha/hora, sector, tipo de incidente, modalidad.
- **Ferretería/ventas:** producto, unidades, precio, stock.
- **Finanzas:** cliente, producto, saldo, mora, fecha de pago.
- **Educación:** estudiante, fecha, asistencia, tardanza, calificación.
- **Logística:** guía, vehículo, fecha, tiempo, estado, destino.

### Preguntas que cada equipo debe responder
1. ¿Qué representaciones de nulos aparecen en nuestro archivo?
2. ¿Cuál es la clave de negocio para reconocer un posible duplicado?
3. ¿Qué reglas hacen que un valor sea imposible en nuestro contexto?
4. ¿Qué transformaciones son de formato y cuáles cambiarían el significado?
5. ¿Qué valores atípicos deben revisarse antes de decidir si son errores?
6. ¿Qué cambios deben quedar registrados en la bitácora?


## Referencias técnicas
- pandas documentation — Missing data, `to_datetime`, `to_numeric`, `duplicated` y `drop_duplicates`.
- Wickham, H. (2014). *Tidy Data*. Journal of Statistical Software, 59(10), 1–23.
- Proyecto Curricular de Ciencia de Datos e Inteligencia Artificial y Plan de Trabajo EFSRT de Proyecto Productivo IB.
